# Day 13 — Advanced Functions

**Intern:** Sehar Andleeb &nbsp;|&nbsp; **Org:** Xeven Solutions &nbsp;|&nbsp; **Repo:** `ai-internship-xeven-2026`

This notebook documents Day 13. It explains the four core concepts, then imports and demonstrates the three task scripts (`flexible_logger.py`, `data_transformer.py`, `comprehensions_toolkit.py`) with live output.

**Concepts covered:** `*args` / `**kwargs`, lambda functions, list comprehensions, dictionary comprehensions.

## 1. Theoretical Concepts

### 1.1 `*args` and `**kwargs`
These let a function accept a *variable* number of arguments.
- `*args` collects extra **positional** arguments into a **tuple**.
- `**kwargs` collects extra **keyword** arguments into a **dict**.
- Use `*args` when the count of values is unknown (e.g. `sum_all(1, 2, 3)`); use `**kwargs` for optional named settings (e.g. `connect(host='db', port=5432)`).
- The same `*`/`**` also *unpack* a list/dict back into arguments: `func(*my_list, **my_dict)`.

### 1.2 Lambda functions
A `lambda` is a small **anonymous** function written in one line: `lambda x: x * 2`. It has no name and returns its single expression automatically. Lambdas shine as throwaway callbacks passed to `map()`, `filter()`, and `sorted(key=...)`. For anything multi-line or reused, prefer a named `def`.

### 1.3 List comprehensions
A compact way to build a list: `[expr for item in iterable if condition]`. It replaces the common *create-empty-list → loop → append* pattern with one readable line, and is usually faster because the looping runs in optimized C.

### 1.4 Dictionary comprehensions
Same idea, producing a dict: `{key: value for item in iterable if condition}`. Great for transforming or filtering mappings, e.g. inverting a dict or counting items.

### Quick examples
The cell below shows each concept in its smallest form before we move to the full task scripts.

In [1]:
# *args / **kwargs
def demo(*args, **kwargs):
    return args, kwargs

print('args/kwargs:', demo(1, 2, mode='fast'))

# lambda inside map / filter / sorted
nums = [5, 2, 9, 1]
print('map    :', list(map(lambda n: n * n, nums)))
print('filter :', list(filter(lambda n: n > 3, nums)))
print('sorted :', sorted(nums, key=lambda n: -n))

# list + dict comprehension
print('list comp:', [n * 2 for n in nums if n % 2])
print('dict comp:', {n: n * n for n in nums})

args/kwargs: ((1, 2), {'mode': 'fast'})
map    : [25, 4, 81, 1]
filter : [5, 9]
sorted : [9, 5, 2, 1]
list comp: [10, 18, 2]
dict comp: {5: 25, 2: 4, 9: 81, 1: 1}


## 2. Practical Implementation

We import the three task modules so the notebook documents and demonstrates the *same* code that lives in the `.py` files (single source of truth).

In [2]:
import importlib
import flexible_logger
import data_transformer as dt
import comprehensions_toolkit as ct

# Reload in case the scripts were edited during the session.
importlib.reload(flexible_logger)
importlib.reload(dt)
importlib.reload(ct)
print('Modules imported successfully.')

Modules imported successfully.


## 3. Practical Tasks

### Task 1 — Flexible Logger Function
`log(level, *messages, **options)` accepts any number of message fragments (`*messages`) plus optional settings (`**options`): `timestamp`, `file`, and `format` (`text` or `json`). Output is color-tinted by level (ERROR=red, WARNING=yellow, INFO=normal).

In [3]:
flexible_logger.log('INFO', 'Application', 'started',
                    timestamp=True)
flexible_logger.log('WARNING', 'Disk', 'space', 'low',
                    timestamp=True)
json_line = flexible_logger.log('ERROR',
                                'Database connection failed',
                                format='json', timestamp=True)
print('Returned value:', json_line)

[2026-06-12T10:59:07] INFO: Application started
[2026-06-12T10:59:07] WARNING: Disk space low
{"level": "ERROR", "message": "Database connection failed", "timestamp": "2026-06-12T10:59:07"}
Returned value: {"level": "ERROR", "message": "Database connection failed", "timestamp": "2026-06-12T10:59:07"}


### Task 2 — Data Transformer Suite
Lambdas inside `map()` (clean strings), `filter()` (extract emails / phones / URLs), and `sorted()` (multi-key sort). Then a performance comparison of lambda vs named function vs list comprehension on the same workload.

In [4]:
messy = ['  hello ', 'World  ', ' PyThOn']
print('Cleaned:', dt.clean_strings(messy))

blobs = ['mail me at a@b.com', 'visit https://x.io',
         'plain text', 'call +1 555 123 4567']
print('Emails:', dt.extract_matches(blobs, dt.EMAIL_RE))
print('URLs  :', dt.extract_matches(blobs, dt.URL_RE))
print('Phones:', dt.extract_matches(blobs, dt.PHONE_RE))

people = [
    {'name': 'Ali', 'age': 30},
    {'name': 'Sara', 'age': 25},
    {'name': 'Bilal', 'age': 30},
]
print('Sorted:', dt.sort_by_keys(people, ['age', 'name']))

Cleaned: ['HELLO', 'WORLD', 'PYTHON']
Emails: ['mail me at a@b.com']
URLs  : ['visit https://x.io']
Phones: ['call +1 555 123 4567']
Sorted: [{'name': 'Sara', 'age': 25}, {'name': 'Ali', 'age': 30}, {'name': 'Bilal', 'age': 30}]


**Performance comparison** (fastest of several runs). The list comprehension is typically fastest because its loop runs in optimized C with no per-item function-call overhead; the lambda and named function are close to each other since both pay that call cost.

In [5]:
numbers = list(range(100_000))
print('lambda + map :  %.5fs'
      % dt.benchmark(dt.square_lambda, numbers))
print('named func   :  %.5fs'
      % dt.benchmark(dt.square_named, numbers))
print('comprehension:  %.5fs'
      % dt.benchmark(dt.square_comprehension, numbers))

lambda + map :  0.00411s
named func   :  0.00413s
comprehension:  0.00208s


### Task 3 — Advanced List/Dict Comprehensions
Flatten a nested list, transpose a matrix (nested comprehension), invert a dict, and build a word-frequency counter (dict comprehension).

In [6]:
print('Flatten  :', ct.flatten([[1, 2], [3, 4], [5]]))
print('Transpose:', ct.transpose([[1, 2, 3], [4, 5, 6]]))
print('Invert   :', ct.invert_dict({'a': 1, 'b': 2}))
print('Frequency:',
      ct.word_frequency('the cat the dog the bird'))

Flatten  : [1, 2, 3, 4, 5]
Transpose: [[1, 4], [2, 5], [3, 6]]
Invert   : {1: 'a', 2: 'b'}
Frequency: {'dog': 1, 'bird': 1, 'cat': 1, 'the': 3}


## 4. Research & References

*Per the roadmap, each concept was consulted across four sources. Replace the bracketed share-links with your own saved URLs before pushing.*

| Source | Consulted | URL | Key takeaway |
|---|---|---|---|
| ChatGPT | June 13, 2026 | `[paste share link]` | `*args`/`**kwargs` framed as "collect extras into a tuple / dict"; clear on the unpack-vs-collect duality of `*`/`**`. |
| Gemini | June 13, 2026 | `[paste share link]` | Strong on *when* to pick a lambda vs `def` — lambdas only for short, throwaway callbacks. |
| Claude | June 13, 2026 | `[paste share link]` | Best on comprehension performance (C-level loop, no per-item call overhead) and readability limits. |
| Article (Real Python / Medium) | June 13, 2026 | `[paste article URL]` | Worked examples of dict comprehension for inversion and frequency counting. |

**Comparison of the four concepts**

| Concept | Returns | Best for | Watch out for |
|---|---|---|---|
| `*args` / `**kwargs` | tuple / dict | Flexible APIs, wrappers, optional settings | Hard-to-read signatures if overused |
| Lambda | a function object | Short inline callbacks in `map`/`filter`/`sorted` | No statements, one expression only |
| List comprehension | a list | Building/filtering lists concisely and fast | Nested comps hurt readability |
| Dict comprehension | a dict | Transforming / inverting / counting mappings | Duplicate keys silently overwrite |

**Clearest Explanation:** Claude — it tied each construct to a concrete *when-to-use-it* rule and explained *why* comprehensions run faster, instead of only showing syntax.

## 5. Summary
Day 13 moved from defining functions to using them flexibly: accepting variable arguments with `*args`/`**kwargs`, writing concise inline logic with lambdas, and transforming data declaratively with list and dict comprehensions. The three scripts apply each idea to a realistic utility — a logger, a data-cleaning suite, and a comprehension toolkit — the kind of small reusable helpers that show up constantly in real data and backend work.